In [ ]:
import numpy as np
import torch
import time
from lqa_basic import Lqa_basic
import cim_optimizer.solve_Ising as cim
import simulated_bifurcation as sb
import dimod


def generate_sk_matrix(N, seed):
    np.random.seed(seed)
    rand_vals = np.random.randn(N, N)
    J = torch.tensor(rand_vals, dtype=torch.float32)
    J = torch.triu(J, diagonal=1)
    J = J + J.T
    J.fill_diagonal_(0)
    return J

def benchmark_lqa_basic(num_instances=1, N=10):
    results = []

    for seed in range(num_instances):
        print(f"\nRunning instance with seed {seed}")

        # Generate SK matrix
        sk_matrix = generate_sk_matrix(N, seed)

        # Initialize solver
        solver = Lqa_basic(sk_matrix)

        # Time-limited optimization
        start_time = time.time()
        solver.minimise(step=0.5, N=1000, g=1, f=0.1, mom=0.99)
        elapsed = time.time() - start_time

        # Enforce 1-second runtime
        if elapsed > 1.0:
            print(f"Warning: Runtime exceeded 1s (took {elapsed:.2f}s)")

        # Record result
        energy = solver.energy/(N**0.5)# Normalizing by sqrt(N) for SK model
        
        print(f"Instance {seed}: Energy = {energy:.4f}, Time = {elapsed:.4f}s")
        results.append((seed, energy, elapsed))

    return results

C:\Users\niyat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
N = 128
seed = 0
J_matrix = generate_sk_matrix(N,seed).numpy()

print("benchmarking methods for N="+str(N)+" spins and seed="+str(seed))
print("")


print("LQA")
results_lqa = benchmark_lqa_basic(1,N)
print("")

print("dimod")
print("")
start = time.time()
# Define the Ising coupling matrix


# Convert to a dict of couplings: {(i, j): J_ij}
J = {}
n = len(J_matrix)
for i in range(n):
    for j in range(i + 1, n):  # only upper triangle, as J_ij = J_ji
        if J_matrix[i][j] != 0:
            J[(i, j)] = J_matrix[i][j]

# No local fields
h = {i: 0 for i in range(n)}

# Create the Ising model (SPIN means ±1 variables)
bqm = dimod.BinaryQuadraticModel(h, J, 0.0, vartype=dimod.SPIN)

# Use simulated annealing sampler
sampler = dimod.SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm, num_reads=15)

# Extract the best result
best_sample = sampleset.first.sample
best_energy = sampleset.first.energy

#print("Best spin configuration (z_i):", best_sample)
print("Minimum Ising energy using dimod:", best_energy)
print("time used by dimod:", time.time()-start)
print("")

print("cim:")
print("")
solution = cim.Ising(J_matrix).solve( hyperparameters_randomtune = False)
print("")

print("simulated bifurcation")
print("")
ising = sb.QuadraticPolynomial(J_matrix/2)
spins, value = ising.minimize(domain='spin')
value
print("")
print("summary of optimal values")
print("")
print("LQA=",results_lqa[0][1]*np.sqrt(N))
print("dimod=",best_energy)
print("cim=",solution.result['lowest_energy'])
print("simulated bifurcation=",value.item())

benchmarking methods for N=128 spins and seed=0

LQA

Running instance with seed 0
min energy -88.77674865722656
Instance 0: Energy = -7.8468, Time = 0.0682s

dimod

Minimum Ising energy using dimod: -90.2485080076151
time used by dimod: 66.72678518295288

cim:

No External Field Detected
Target Ising Energy: -inf.
Best Ising Energy Found: -68.15641021728516.
Corresponding Spin Configuration: [ 1.  1. -1.  1. -1.  1. -1. -1.  1.  1.  1. -1. -1. -1.  1.  1.  1.  1.
  1. -1. -1. -1. -1. -1. -1. -1.  1. -1. -1. -1.  1. -1. -1. -1. -1. -1.
 -1. -1.  1. -1.  1.  1.  1.  1. -1. -1. -1. -1.  1. -1.  1.  1.  1. -1.
  1. -1. -1. -1. -1. -1. -1.  1.  1. -1. -1.  1.  1. -1. -1.  1. -1. -1.
 -1.  1. -1. -1.  1.  1. -1. -1. -1.  1.  1.  1.  1. -1.  1. -1.  1.  1.
  1.  1. -1.  1.  1.  1. -1. -1.  1. -1.  1.  1.  1. -1. -1.  1. -1. -1.
  1. -1.  1.  1. -1.  1. -1.  1. -1. -1.  1.  1. -1.  1.  1.  1.  1.  1.
  1.  1.].
Time Elapsed: 0.2632110118865967.
Number of Runs Completed: 1.

simulated bifurcat

🏁 Bifurcated agents: 100%|██████████| 128/128 [00:01<00:00, 104.62 agents/s]


summary of optimal values

LQA= -88.77674865722656
dimod= -90.2485080076151
cim= -68.15641021728516
simulated bifurcation= -90.24851989746094
